# EuroSAT 6-Band Land-Use Classification with MobileNetV3

Lightweight satellite image classification for **CubeSat / edge** deployment. A MobileNetV3-Large backbone is adapted from 3 to **6 Sentinel-2 spectral channels** and fine-tuned on the EuroSAT multispectral dataset using a **fixed, reproducible train/val/test split**.

| Section | Content |
|---|---|
| 1 | Environment & dependencies setup |
| 2 | Custom dataset pipeline (`EuroSat6BandDataset`, fixed split in `split.json`) |
| 3 | Model definition (6-channel MobileNetV3) |
| 4 | Training & validation loop (early stopping, loss/accuracy plots) |
| 5 | Model evaluation (confusion matrix, test accuracy & F1) |
| 6 | Inference example (`best_model.pth` on single images) |

> The reusable code lives in `dataset.py`, `model.py`, `train.py` and `infer.py`; this notebook imports from them so there is a single source of truth.

## 1. Environment & Dependencies Setup

In [ ]:
# Uncomment to install dependencies (e.g. on Google Colab). Locally: pip install -r requirements.txt
# !pip install -q tifffile scikit-learn matplotlib numpy

In [ ]:
import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

**Configuration.** Point `DATA_DIR` at the folder that contains one sub-folder per class (`AnnualCrop/`, `Forest/`, ...) filled with 13-band `.tif` tiles from the EuroSAT *all bands* release.

In [ ]:
# ---- Edit these paths -------------------------------------------------------
DATA_DIR   = os.environ.get("EUROSAT_DIR", "data/EuroSATallBands/ds/images/remote_sensing/otherDatasets/sentinel_2/tif")
SPLIT_FILE = "split.json"        # fixed train/val/test indices (created on first run if missing)
OUTPUT_DIR = "outputs"           # best_model.pth, plots and metrics are written here
# -----------------------------------------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT = os.path.join(OUTPUT_DIR, "best_model.pth")

*(Optional)* Download and unpack the dataset (about 2 GB). Skip if you already have it.

In [ ]:
# !wget -q https://madm.dfki.de/files/sentinel/EuroSATallBands.zip -O EuroSATallBands.zip
# !unzip -q EuroSATallBands.zip -d data/EuroSATallBands

## 2. Custom Dataset Pipeline

`EuroSat6BandDataset` reads the 13-band GeoTIFFs, keeps **6 bands**, scales reflectance by `1/10000` and serves one of the **fixed splits** stored in `split.json` (80 % train / 10 % val / 10 % test, seed 42). Using a saved split guarantees every run is evaluated on the same held-out test images. Training samples receive geometry- and radiometry-preserving augmentation (flips, 90° rotations, noise, per-band gain).

In [ ]:
from dataset import EuroSat6BandDataset, get_dataloaders, SELECTED_BANDS, CLASS_NAMES

print("Selected band indices (0-based, into the 13-band tiles):", SELECTED_BANDS)

loaders, classes = get_dataloaders(DATA_DIR, split_file=SPLIT_FILE, batch_size=32, num_workers=2, seed=SEED)
for name, loader in loaders.items():
    print(f"{name:>5}: {len(loader.dataset):>6} images, {len(loader):>4} batches")
print("Classes:", classes)

In [ ]:
# Visual sanity check: true-colour previews (RGB = channels [2, 1, 0] -> B4, B3, B2) of test tiles.
preview = EuroSat6BandDataset(DATA_DIR, split="test", split_file=SPLIT_FILE, augment=False)
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for ax, i in zip(axes.ravel(), np.linspace(0, len(preview) - 1, 10, dtype=int)):
    x, y = preview[i]
    rgb = x[[2, 1, 0]].permute(1, 2, 0).numpy()
    ax.imshow((rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6))
    ax.set_title(classes[y], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Model Definition

`MobileNetV3Classifier` is an ImageNet-pretrained **MobileNetV3-Large** whose first convolution is widened from 3 to 6 input channels (transfer learning):

* pretrained RGB filters -> first 3 channels,
* mean RGB filter -> the 3 extra channels (B8, plus the two SWIR channels),
* ImageNet head replaced by a compact MLP (`960 -> 256 -> 128 -> 10`) with dropout 0.67 / 0.47.

The first call downloads the torchvision ImageNet weights.

In [ ]:
from model import MobileNetV3Classifier, count_parameters

model = MobileNetV3Classifier(num_bands=6, num_classes=len(classes), pretrained=True).to(device)
print(f"Trainable parameters: {count_parameters(model):,}")

with torch.no_grad():
    out = model.eval()(torch.randn(2, 6, 64, 64, device=device))
print("Output shape:", tuple(out.shape))  # (batch, 10)

## 4. Training & Validation Loop

Settings: Adam (`lr=1e-4`, `weight_decay=1e-3`), cross-entropy with label smoothing 0.1, gradient clipping at 1.0, `ReduceLROnPlateau` on validation loss, **early stopping** on validation loss (patience 7), and the checkpoint with the **best validation accuracy** is written to `best_model.pth`. The epoch loop is implemented in `train.fit()`.

In [ ]:
from train import fit, plot_history

NUM_EPOCHS = 10
history, best_val_acc, criterion = fit(
    model, loaders, device,
    epochs=NUM_EPOCHS, lr=1e-4, weight_decay=1e-3,
    label_smoothing=0.1, patience=7, checkpoint_path=CHECKPOINT,
)
print(f"\nBest validation accuracy: {best_val_acc:.2f}%  ->  saved to {CHECKPOINT}")

In [ ]:
plot_history(history, save_path=os.path.join(OUTPUT_DIR, "training_curves.png"), show=True)

> **Note on the curves:** training accuracy is measured *with* augmentation and 0.67/0.47 dropout active, so it is normally *lower* than validation accuracy (which uses clean images, dropout off). A negative train-val gap is expected here and is not a bug.

## 5. Model Evaluation

The best checkpoint is evaluated **once** on the held-out test split. Metrics are support-weighted.

Reference result from the original training run (10 epochs, 2,700 test images):

| Split | Loss | Accuracy | F1 | Recall | Precision |
|---|---|---|---|---|---|
| Validation (best epoch 7) | 0.5805 | 97.11 % | 97.11 % | 97.11 % | 97.13 % |
| **Test** | 0.5763 | **97.37 %** | **97.37 %** | 97.37 % | 97.38 % |

Re-running will give numbers close to, but not necessarily identical to, these (GPU nondeterminism, augmentation randomness and — if you regenerate it — a different split).

In [ ]:
from train import evaluate, plot_confusion_matrix
from sklearn.metrics import classification_report

model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
test_metrics, y_true, y_pred = evaluate(model, loaders["test"], criterion, device, return_predictions=True)

print(f"TEST | loss {test_metrics['loss']:.4f} | accuracy {test_metrics['acc']:.2f}% | F1 {test_metrics['f1']:.2f}% "
      f"| recall {test_metrics['recall']:.2f}% | precision {test_metrics['precision']:.2f}%")

In [ ]:
cm = plot_confusion_matrix(y_true, y_pred, classes, save_path=os.path.join(OUTPUT_DIR, "confusion_matrix.png"), show=True)

In [ ]:
print(classification_report(y_true, y_pred, target_names=classes, digits=4, zero_division=0))

## 6. Inference Example

Load `best_model.pth` and classify individual GeoTIFF tiles. `load_checkpoint` rebuilds the architecture without downloading ImageNet weights. The same logic is available from the command line:

```bash
python infer.py --weights outputs/best_model.pth path/to/tile.tif
```

In [ ]:
from model import load_checkpoint
from infer import predict

net = load_checkpoint(CHECKPOINT, device, num_bands=6, num_classes=len(CLASS_NAMES))

test_ds = EuroSat6BandDataset(DATA_DIR, split="test", split_file=SPLIT_FILE, augment=False)
sample_ids = [0, 1, 2, 3]

fig, axes = plt.subplots(1, len(sample_ids), figsize=(3.2 * len(sample_ids), 3.6))
for ax, i in zip(axes, sample_ids):
    path, true_label = test_ds.image_paths[i], CLASS_NAMES[test_ds.labels[i]]
    top3, tensor = predict(net, path, device, top_k=3)
    rgb = tensor[[2, 1, 0]].permute(1, 2, 0).numpy()
    ax.imshow((rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)); ax.axis("off")
    ax.set_title(f"True: {true_label}\nPred: {top3[0][0]} ({top3[0][1]*100:.1f}%)", fontsize=9)
    print(f"{os.path.basename(path)}: " + " | ".join(f"{n} {p*100:.1f}%" for n, p in top3))
plt.tight_layout(); plt.show()